<a href="https://colab.research.google.com/github/nina-pham/fragmentation-replication/blob/main/Fragmentation_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install dependencies/mount Google Drive

In [ ]:
!pip install duckdb==0.10.3 --quiet

import duckdb
import os
import zipfile
import pandas as pd
from datetime import datetime

print(f"DuckDB version : {duckdb.__version__}")
print(f"Pandas version : {pd.__version__}")
print(f"Run timestamp  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.9 MB/s eta 0:00:00
DuckDB version : 0.10.3
Pandas version : 2.2.2
Run timestamp  : 2026-04-02 13:31:32


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Google Drive mounted at /content/drive")

Mounted at /content/drive
Google Drive mounted at /content/drive


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CONFIG BLOCK
# ════════════════════════════════════════════════════════════════════════════════

# Google Drive paths
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/CSIS"

# File paths
ZIP_FILENAME = "_TDM_HS4_2015_2024.zip"
TSV_FILENAME = "TDM_report.csv"

ZIP_PATH = os.path.join(DRIVE_ROOT, ZIP_FILENAME)
EXTRACT_DIR = "/content/tdm_data"
TSV_PATH = os.path.join(EXTRACT_DIR, TSV_FILENAME)
DB_PATH = "/content/tdm.duckdb"

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Input zip: {ZIP_PATH}")
print(f"Extract dir: {EXTRACT_DIR}")
print(f"TSV path: {TSV_PATH}")
print(f"DB path: {DB_PATH}")

CONFIGURATION
Drive root: /content/drive/MyDrive/Colab Notebooks/CSIS
Input zip: /content/drive/MyDrive/Colab Notebooks/CSIS/_TDM_HS4_2015_2024.zip
Extract dir: /content/tdm_data
TSV path: /content/tdm_data/TDM_report.csv
DB path: /content/tdm.duckdb


## 2. Extract & Load Data

In [ ]:
import os, zipfile

os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(TSV_PATH):
    print(f"Extracting {ZIP_PATH} → {EXTRACT_DIR} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extraction complete.")
else:
    print(f"Already extracted: {TSV_PATH}")

# List extracted files
for f in os.listdir(EXTRACT_DIR):
    fpath = os.path.join(EXTRACT_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

Extracting /content/drive/MyDrive/Colab Notebooks/CSIS/_TDM_HS4_2015_2024.zip → /content/tdm_data ...
Extraction complete.
  202602171920111530_report.csv  (9846.1 MB)


In [ ]:
import duckdb
import os # Added import for os module

con = duckdb.connect(DB_PATH)

# Check if the configured TSV_PATH exists. If not, try to find the actual CSV file.
if not os.path.exists(TSV_PATH):
    print(f"Warning: Configured TSV_PATH '{TSV_PATH}' not found.")
    csv_files = [f for f in os.listdir(EXTRACT_DIR) if f.endswith('.csv')]
    if csv_files:
        # Assuming the first CSV file found is the correct one.
        # This is a heuristic and might need adjustment if multiple CSVs are expected.
        TSV_FILENAME_ACTUAL = csv_files[0]
        TSV_PATH = os.path.join(EXTRACT_DIR, TSV_FILENAME_ACTUAL)
        print(f"Using detected TSV_PATH: '{TSV_PATH}'")
    else:
        raise FileNotFoundError(f"No CSV files found in {EXTRACT_DIR} to process.")

# Try tab-delimited first (standard COMTRADE TDM format),
# fall back to comma-delimited
try:
    sample = con.execute(f"""
        SELECT * FROM read_csv_auto(
            '{TSV_PATH}',
            delim = '\t',
            -- encoding = 'UTF-16', -- Removed: 'encoding' is not a valid parameter for read_csv_auto
            sample_size = 1000
        ) LIMIT 5
    """).df()
    ENCODING = 'UTF-16' # This variable is used in subsequent cells for `read_csv` configuration.
    DELIM = '\t'
    print("✓ Detected: UTF-16, tab-delimited")
except Exception as e: # Catch broader exception to handle various failures including parsing
    print(f"Attempt 1 (tab-delimited) failed: {e}")
    try:
        sample = con.execute(f"""
            SELECT * FROM read_csv_auto(
                '{TSV_PATH}',
                sample_size = 1000
            ) LIMIT 5
        """).df()
        ENCODING = 'UTF-8'
        DELIM = ','
        print("✓ Detected: UTF-8, auto-delimited")
    except Exception as e:
        print(f"✗ Could not parse file: {e}")
        raise

print(f"\nColumns ({len(sample.columns)}):")
for i, col in enumerate(sample.columns):
    print(f"  [{i:2d}] {col}")

print("\nFirst 3 rows:")
sample.head(3)

✗ Could not parse file: IO Error: No files found that match the pattern "/content/tdm_data/TDM_report.csv"


IOException: IO Error: No files found that match the pattern "/content/tdm_data/TDM_report.csv"

In [ ]:
#
# After running Cell 5, update the mapping below to match column names.
# The pipeline uses these standardised names internally.
#
# If your column names already match the LEFT side, no changes needed.

In [ ]:
con = duckdb.connect(DB_PATH)

# --- Raw view: reads directly from TSV with proper encoding ---
if ENCODING == 'UTF-16' and DELIM == '\t':
    read_expr = f"""
        read_csv_auto(
            '{TSV_PATH}',
            delim = '\t',
            encoding = 'UTF-16',
            sample_size = 100000
        )
    """
else:
    read_expr = f"""
        read_csv_auto('{TSV_PATH}', sample_size = 100000)
    """

con.execute(f"CREATE OR REPLACE VIEW tdm_raw AS SELECT * FROM {read_expr}")

# --- Clean view: standardised column names, 2023 only, HS4 ---
col_selects = []
for std_name, raw_name in COLUMN_MAP.items():
    col_selects.append(f'"{raw_name}" AS {std_name}')

select_clause = ",\n    ".join(col_selects)

con.execute(f"""
    CREATE OR REPLACE VIEW tdm_clean AS
    SELECT
        {select_clause}
    FROM tdm_raw
    WHERE "{COLUMN_MAP['year']}" = {TARGET_YEAR}
      AND LENGTH(CAST("{COLUMN_MAP['commodity_code']}" AS VARCHAR)) = 4
""")

# Quick check
row_count = con.execute("SELECT COUNT(*) FROM tdm_clean").fetchone()[0]
print(f"✓ tdm_clean view created: {row_count:,} rows (year={TARGET_YEAR}, HS4 only)")

# Preview
con.execute("SELECT * FROM tdm_clean LIMIT 5").df()

## 3. Validation

In [ ]:
print("=" * 60)
print("VALIDATION CHECKS")
print("=" * 60)

# Check 1: Row count
n = con.execute("SELECT COUNT(*) FROM tdm_clean").fetchone()[0]
print(f"\n[1] Row count: {n:,}")

# Check 2: Year coverage (should only be 2023)
years = con.execute("SELECT DISTINCT year FROM tdm_clean ORDER BY year").df()
print(f"[2] Years present: {list(years['year'])}")

# Check 3: Flow balance (imports vs exports)
flow_counts = con.execute("""
    SELECT trade_flow_code, COUNT(*) as n, SUM(trade_value) as total_value
    FROM tdm_clean
    GROUP BY trade_flow_code
    ORDER BY trade_flow_code
""").df()
print(f"[3] Flow balance:")
for _, row in flow_counts.iterrows():
    label = "Import" if row['trade_flow_code'] == FLOW_IMPORT else "Export"
    print(f"    {label} (code={int(row['trade_flow_code'])}): "
          f"{int(row['n']):,} rows, ${row['total_value']/1e9:.1f}B")

# Check 4: HS4 consistency (all codes should be 4 digits)
hs4_check = con.execute("""
    SELECT LENGTH(CAST(commodity_code AS VARCHAR)) as code_len,
           COUNT(*) as n
    FROM tdm_clean
    GROUP BY code_len
    ORDER BY code_len
""").df()
print(f"[4] HS4 code lengths: {dict(zip(hs4_check['code_len'], hs4_check['n']))}")

# Check 5: Total trade value
totals = con.execute("""
    SELECT SUM(trade_value) as total_usd FROM tdm_clean
""").fetchone()[0]
print(f"[5] Total trade value: ${totals/1e12:.2f} trillion")

# Check 6: Top 10 reporters by value
top_reporters = con.execute("""
    SELECT reporter_iso, SUM(trade_value) as total
    FROM tdm_clean
    GROUP BY reporter_iso
    ORDER BY total DESC
    LIMIT 10
""").df()
print(f"[6] Top 10 reporters: {list(top_reporters['reporter_iso'])}")

# Check 7: Non-standard partner codes (World=0, unspecified, etc.)
non_standard = con.execute("""
    SELECT partner_code, partner_iso, COUNT(*) as n
    FROM tdm_clean
    WHERE partner_code IN (0, 97, 98, 99, 896, 899)
       OR partner_iso IN ('WLD', 'UNS', '')
       OR partner_iso IS NULL
    GROUP BY partner_code, partner_iso
    ORDER BY n DESC
""").df()
print(f"[7] Non-standard partner codes: {len(non_standard)} types, "
      f"{non_standard['n'].sum():,} rows")
if len(non_standard) > 0:
    print(non_standard.to_string(index=False))

# Check 8: Null/zero value check
nulls = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE trade_value IS NULL) as null_values,
        COUNT(*) FILTER (WHERE trade_value = 0) as zero_values,
        COUNT(*) FILTER (WHERE trade_value < 0) as negative_values
    FROM tdm_clean
""").df()
print(f"[8] Value quality: nulls={int(nulls['null_values'][0])}, "
      f"zeros={int(nulls['zero_values'][0])}, "
      f"negatives={int(nulls['negative_values'][0])}")

# Check 9: Unique reporters and partners
n_reporters = con.execute("SELECT COUNT(DISTINCT reporter_iso) FROM tdm_clean").fetchone()[0]
n_partners = con.execute("SELECT COUNT(DISTINCT partner_iso) FROM tdm_clean").fetchone()[0]
n_products = con.execute("SELECT COUNT(DISTINCT commodity_code) FROM tdm_clean").fetchone()[0]
print(f"[9] Unique: {n_reporters} reporters, {n_partners} partners, {n_products} HS4 products")

# Check 10: Duplicate check (same reporter-partner-flow-product)
dupes = con.execute("""
    SELECT COUNT(*) as n_dupes FROM (
        SELECT reporter_iso, partner_iso, trade_flow_code, commodity_code,
               COUNT(*) as cnt
        FROM tdm_clean
        GROUP BY reporter_iso, partner_iso, trade_flow_code, commodity_code
        HAVING cnt > 1
    )
""").fetchone()[0]
print(f"[10] Duplicate flows: {dupes:,}")

# Check 11: Sample of data
print(f"\n[11] Sample rows:")
con.execute("SELECT * FROM tdm_clean ORDER BY trade_value DESC LIMIT 5").df()

## 4. GeoDist & Country Metadata

In [ ]:
import pandas as pd
import os

# --- Try loading from Google Drive first, then download from CEPII ---
GEODIST_DRIVE_PATH = os.path.join(DRIVE_ROOT, "geo_cepii.csv")
GEODIST_LOCAL_PATH = "/content/geo_cepii.csv"
GEODIST_URL = "http://www.cepii.fr/distance/geo_cepii.csv"

# Also need the bilateral distance file (dist_cepii)
DIST_DRIVE_PATH = os.path.join(DRIVE_ROOT, "dist_cepii.csv")
DIST_LOCAL_PATH = "/content/dist_cepii.csv"
DIST_URL = "http://www.cepii.fr/distance/dist_cepii.csv"

def load_or_download(drive_path, local_path, url, name):
    """Try Drive first, then local cache, then download."""
    if os.path.exists(drive_path):
        print(f"✓ {name}: loaded from Drive")
        return pd.read_csv(drive_path)
    elif os.path.exists(local_path):
        print(f"✓ {name}: loaded from local cache")
        return pd.read_csv(local_path)
    else:
        print(f"↓ {name}: downloading from CEPII...")
        try:
            df = pd.read_csv(url)
            df.to_csv(local_path, index=False)
            # Also save to Drive for future runs
            try:
                df.to_csv(drive_path, index=False)
                print(f"  → saved to Drive for future runs")
            except:
                pass
            print(f"✓ {name}: downloaded ({len(df):,} rows)")
            return df
        except Exception as e:
            print(f"✗ {name}: download failed — {e}")
            print(f"  Please manually download from {url}")
            print(f"  and place it at: {drive_path}")
            return None

# The bilateral distance dataset is what we need for gravity variables
dist_df = load_or_download(DIST_DRIVE_PATH, DIST_LOCAL_PATH, DIST_URL, "dist_cepii")

if dist_df is not None:
    print(f"\ndist_cepii columns: {list(dist_df.columns)}")
    print(f"Shape: {dist_df.shape}")
    print(f"\nKey columns for gravity regression:")
    # Standard dist_cepii columns:
    # iso_o, iso_d, dist, distcap, distw, distwces,
    # contig, comlang_off, comlang_ethno, colony,
    # comcol, curcol, col45, smctry
    for col in ['dist', 'distcap', 'contig', 'comlang_off', 'colony']:
        if col in dist_df.columns:
            print(f"  {col}: {dist_df[col].describe().to_dict()}")
    dist_df.head(3)

In [ ]:
# Install pycountry if needed
import subprocess
try:
    import pycountry
except ImportError:
    subprocess.run(["pip", "install", "pycountry", "-q"])
    import pycountry

# Build ISO3 → ISO2 lookup
iso3_to_iso2 = {}
for c in pycountry.countries:
    iso3_to_iso2[c.alpha_3] = c.alpha_2

print(f"✓ ISO3→ISO2 mapping: {len(iso3_to_iso2)} countries")

# Landlocked countries (from CEPII/World Bank classification)
LANDLOCKED = {
    "AFG", "AND", "ARM", "AUT", "AZE", "BLR", "BTN", "BOL", "BWA",
    "BFA", "BDI", "CAF", "TCD", "CZE", "ETH", "HUN", "KAZ", "KGZ",
    "LAO", "LSO", "LIE", "LUX", "MKD", "MWI", "MLI", "MDA", "MNG",
    "NPL", "NER", "PRY", "RWA", "SRB", "SVK", "SSD", "SWZ", "CHE",
    "TJK", "TKM", "UGA", "UZB", "ZMB", "ZWE",
}
print(f"✓ Landlocked set: {len(LANDLOCKED)} countries")

# Continent mapping from dist_cepii (if available) or manual
# We'll extract this from GeoDist's geo_cepii if possible
geo_df = load_or_download(
    os.path.join(DRIVE_ROOT, "geo_cepii.csv"),
    "/content/geo_cepii.csv",
    "http://www.cepii.fr/distance/geo_cepii.csv",
    "geo_cepii"
)

country_continent = {}
if geo_df is not None and 'iso3' in geo_df.columns and 'continent' in geo_df.columns:
    country_continent = dict(zip(geo_df['iso3'], geo_df['continent']))
    print(f"✓ Continent mapping: {len(country_continent)} countries")
else:
    print("⚠ geo_cepii not available or missing columns; continent mapping will be limited")
    print("  Tiered adjustment will fall back to distance-based tier assignment")

## 5. Mirror Pair Construction

In [ ]:
print("Building mirror pairs...")
print("(joining each export report with its corresponding import report)\n")

# Clean out non-standard partners (World aggregates, unspecified, etc.)
con.execute("""
    CREATE OR REPLACE VIEW tdm_bilateral AS
    SELECT *
    FROM tdm_clean
    WHERE partner_iso IS NOT NULL
      AND partner_iso != ''
      AND partner_code NOT IN (0)
      AND trade_value > 0
""")

bilateral_n = con.execute("SELECT COUNT(*) FROM tdm_bilateral").fetchone()[0]
print(f"Bilateral flows (excl. World/null partners, zero values): {bilateral_n:,}")

# --- Split into exports and imports ---
con.execute("""
    CREATE OR REPLACE VIEW exports AS
    SELECT
        reporter_iso   AS exporter,
        partner_iso    AS importer,
        commodity_code AS hs4,
        trade_value    AS export_value,
        netweight      AS export_weight
    FROM tdm_bilateral
    WHERE trade_flow_code = {FLOW_EXPORT}
""".format(FLOW_EXPORT=FLOW_EXPORT*i

con.execute("""
    CREATE OR REPLACE VIEW imports AS
    SELECT
        partner_iso    AS exporter,       -- mirror: the partner is the exporter
        reporter_iso   AS importer,       -- mirror: the reporter is the importer
        commodity_code AS hs4,
        trade_value    AS import_value,
        netweight      AS import_weight
    FROM tdm_bilateral
    WHERE trade_flow_code = {FLOW_IMPORT}
""".format(FLOW_IMPORT=FLOW_IMPORT))

n_exp = con.execute("SELECT COUNT(*) FROM exports").fetchone()[0]
n_imp = con.execute("SELECT COUNT(*) FROM imports").fetchone()[0]
print(f"Export reports: {n_exp:,}")
print(f"Import reports: {n_imp:,}")

# --- Join to create mirror pairs ---
# A mirror pair exists when BOTH the exporter and importer report the same flow
con.execute("""
    CREATE OR REPLACE TABLE mirror_pairs AS
    SELECT
        COALESCE(e.exporter, i.exporter)  AS exporter,
        COALESCE(e.importer, i.importer)  AS importer,
        COALESCE(e.hs4, i.hs4)            AS hs4,
        e.export_value,
        e.export_weight,
        i.import_value,
        i.import_weight,
        -- Mirror ratio: import_value / export_value (= approx CIF/FOB ratio)
        CASE WHEN e.export_value > 0 AND i.import_value > 0
             THEN i.import_value / e.export_value
             ELSE NULL
        END AS mirror_ratio,
        -- Unit values (value per kg) for gravity regression
        CASE WHEN e.export_weight > 0
             THEN e.export_value / e.export_weight
             ELSE NULL
        END AS uv_export,
        CASE WHEN i.import_weight > 0
             THEN i.import_value / i.import_weight
             ELSE NULL
        END AS uv_import
    FROM exports e
    FULL OUTER JOIN imports i
        ON  e.exporter = i.exporter
        AND e.importer = i.importer
        AND e.hs4      = i.hs4
""")

n_mirror = con.execute("SELECT COUNT(*) FROM mirror_pairs").fetchone()[0]
n_both = con.execute("""
    SELECT COUNT(*) FROM mirror_pairs
    WHERE export_value IS NOT NULL AND import_value IS NOT NULL
""").fetchone()[0]
n_export_only = con.execute("""
    SELECT COUNT(*) FROM mirror_pairs WHERE import_value IS NULL
""").fetchone()[0]
n_import_only = con.execute("""
    SELECT COUNT(*) FROM mirror_pairs WHERE export_value IS NULL
""").fetchone()[0]

print(f"\n✓ Mirror pairs table created: {n_mirror:,} total")
print(f"  Both reports available:  {n_both:,} ({100*n_both/n_mirror:.1f}%)")
print(f"  Export-only (no mirror): {n_export_only:,} ({100*n_export_only/n_mirror:.1f}%)")
print(f"  Import-only (no mirror): {n_import_only:,} ({100*n_import_only/n_mirror:.1f}%)")

# Mirror ratio statistics (for flows with both reports)
ratio_stats = con.execute("""
    SELECT
        COUNT(mirror_ratio) AS n,
        AVG(mirror_ratio) AS mean_ratio,
        MEDIAN(mirror_ratio) AS median_ratio,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY mirror_ratio) AS p25,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY mirror_ratio) AS p75
    FROM mirror_pairs
    WHERE mirror_ratio IS NOT NULL
      AND mirror_ratio BETWEEN 0.1 AND 10   -- exclude extreme outliers for summary
""").df()
print(f"\nMirror ratio (import/export) for matched pairs:")
print(f"  Median: {ratio_stats['median_ratio'][0]:.3f}")
print(f"  Mean:   {ratio_stats['mean_ratio'][0]:.3f}")
print(f"  IQR:    [{ratio_stats['p25'][0]:.3f}, {ratio_stats['p75'][0]:.3f}]")
print(f"  (ratio > 1 means import value exceeds export value, as expected with CIF)")

## 6. Prong 1 — Tiered CIF Adjustment

In [ ]:
#
# NOTE: This tiered approach is NOT from the CEPII/BACI paper.
# It is a pragmatic interim method to unblock the pipeline while the
# gravity regression (Prong 2) is being validated. The tiers are based
# on geographic proximity as a rough proxy for transport costs.

## 7. Prong 2 — Gravity Regression (BACI)

In [ ]:
#
# Implements Equation (1) from Gaulier & Zignago (2010):
#
#   ln(CIF_rate_ij^k) = α + β·ln(dist_ij) + χ·ln(dist_ij)²
#                      + δ·contig_ij + φ·landlocked_i + γ·landlocked_j
#                      + η·ln(UV^k) + ε
#
# where CIF_rate = import_value / export_value for matched mirror pairs

In [ ]:
print("Generating predicted CIF rates for ALL mirror pairs...\n")

# --- Step 1: Build prediction dataset for all mirror pairs ---
# (not just those with observed mirror ratios — we want to predict for all)

predict_df = con.execute(f"""
    SELECT
        mp.exporter,
        mp.importer,
        mp.hs4,
        mp.export_value,
        mp.import_value,
        g.{dist_col} AS dist,
        LN(g.{dist_col}) AS ln_dist,
        POWER(LN(g.{dist_col}), 2) AS ln_dist_sq,
        COALESCE(g.contig, 0) AS contig,
        CASE WHEN mp.exporter IN ({','.join(["'" + c + "'" for c in LANDLOCKED])})
             THEN 1 ELSE 0 END AS landlocked_exp,
        CASE WHEN mp.importer IN ({','.join(["'" + c + "'" for c in LANDLOCKED])})
             THEN 1 ELSE 0 END AS landlocked_imp,
        COALESCE(puv.median_uv, overall_uv.med_uv) AS median_uv,
        LN(COALESCE(puv.median_uv, overall_uv.med_uv)) AS ln_uv
    FROM mirror_pairs mp
    LEFT JOIN geodist g
        ON mp.exporter = g.{iso_o_col}
       AND mp.importer = g.{iso_d_col}
    LEFT JOIN product_uv puv
        ON mp.hs4 = puv.hs4
    CROSS JOIN (SELECT MEDIAN(median_uv) AS med_uv FROM product_uv) overall_uv
    WHERE g.{dist_col} IS NOT NULL
      AND g.{dist_col} > 0
      AND (mp.import_value IS NOT NULL OR mp.export_value IS NOT NULL)
""").df()

print(f"Prediction dataset: {len(predict_df):,} flows")

# --- Step 2: Generate predicted CIF rates ---
X_pred = predict_df[X_cols].copy()
X_pred = sm.add_constant(X_pred)

# Handle any NaN in prediction features (fill with median)
for col in X_cols:
    if X_pred[col].isna().any():
        med_val = X_pred[col].median()
        X_pred[col] = X_pred[col].fillna(med_val)

predict_df['ln_cif_predicted'] = model_clean.predict(X_pred)
predict_df['cif_rate_predicted'] = np.exp(predict_df['ln_cif_predicted'])

# Clip predicted CIF rates to reasonable range
predict_df['cif_rate_predicted'] = predict_df['cif_rate_predicted'].clip(1.0, 1.50)

print(f"\nPredicted CIF rate distribution:")
print(predict_df['cif_rate_predicted'].describe())

# --- Step 3: Apply FOB adjustment ---
predict_df['import_value_fob_gravity'] = predict_df.apply(
    lambda r: r['import_value']  # already FOB for FOB reporters
        if r['importer'] in FOB_REPORTERS
        else (r['import_value'] / r['cif_rate_predicted']
              if pd.notna(r['import_value'])
              else None),
    axis=1
)

print(f"\nFOB adjustment applied.")
print(f"  FOB reporters (no adjustment): {FOB_REPORTERS}")
print(f"  Flows with FOB-adjusted imports: "
      f"{predict_df['import_value_fob_gravity'].notna().sum():,}")

In [ ]:
print("=" * 60)
print("PRONG 2: RECONCILIATION WITH RELIABILITY WEIGHTS")
print("=" * 60)

# --- Step 1: Compute reporting reliability (variance decomposition) ---
# Per BACI Eq (4-5): RD_ij^kt = α_i + β_j + λ_t + γ_k + ε
# We decompose the log-ratio of mirror flows to estimate reporter/partner effects
#
# Since we're working with 2023 only (single year), we drop the time effect
# and estimate: ln(Xij/Mij_fob) = α_i + β_j + γ_k + ε
# where α_i = exporter reliability, β_j = importer reliability

# Build the log-ratio dataset for reliability estimation
predict_df['log_ratio'] = np.where(
    (predict_df['export_value'] > 0) &
    (predict_df['import_value_fob_gravity'] > 0),
    np.log(predict_df['export_value'] / predict_df['import_value_fob_gravity']),
    np.nan
)

reliability_df = predict_df.dropna(subset=['log_ratio']).copy()

# Filter out extreme log-ratios (beyond ±2, i.e. ratio > 7x or < 0.14x)
reliability_df = reliability_df[reliability_df['log_ratio'].abs() <= 2.0]
print(f"Reliability estimation sample: {len(reliability_df):,} paired flows")

# --- Estimate reporter variance components ---
# Exporter variance: how much does each exporter's reporting deviate?
exp_var = reliability_df.groupby('exporter')['log_ratio'].agg(['var', 'count'])
exp_var.columns = ['var_exp', 'n_exp']
exp_var = exp_var[exp_var['n_exp'] >= 10]  # need enough observations

# Importer variance: how much does each importer's reporting deviate?
imp_var = reliability_df.groupby('importer')['log_ratio'].agg(['var', 'count'])
imp_var.columns = ['var_imp', 'n_imp']
imp_var = imp_var[imp_var['n_imp'] >= 10]

# Use median variance as fallback for countries with too few observations
median_var_exp = exp_var['var_exp'].median()
median_var_imp = imp_var['var_imp'].median()

print(f"  Exporter variance estimated for {len(exp_var)} countries (median: {median_var_exp:.4f})")
print(f"  Importer variance estimated for {len(imp_var)} countries (median: {median_var_imp:.4f})")

# --- Step 2: Compute reconciliation weights (BACI Eq 3) ---
# w = exp(σ²_j)·(exp(σ²_j) - 1) / [exp(σ²_i)·(exp(σ²_i) - 1) + exp(σ²_j)·(exp(σ²_j) - 1)]
# where σ²_i = exporter variance, σ²_j = importer variance
# w is the weight on the EXPORT value (higher w = exporter is more reliable)

def compute_baci_weight(var_exporter, var_importer):
    """
    Compute BACI reconciliation weight on export value.
    Per Eq (3): w = g(σ²_j) / [g(σ²_i) + g(σ²_j)]
    where g(σ²) = exp(σ²)(exp(σ²) - 1)  (variance of lognormal)
    Higher w means more weight on the export report.
    """
    g_exp = np.exp(var_exporter) * (np.exp(var_exporter) - 1)
    g_imp = np.exp(var_importer) * (np.exp(var_importer) - 1)

    denom = g_exp + g_imp
    if denom == 0:
        return 0.5  # equal weight if both are zero variance
    return g_imp / denom  # weight on exporter


# Merge variances into prediction dataset
predict_df = predict_df.merge(
    exp_var[['var_exp']].reset_index().rename(columns={'exporter': 'exporter'}),
    on='exporter', how='left'
)
predict_df = predict_df.merge(
    imp_var[['var_imp']].reset_index().rename(columns={'importer': 'importer'}),
    on='importer', how='left'
)

# Fill missing variances with median
predict_df['var_exp'] = predict_df['var_exp'].fillna(median_var_exp)
predict_df['var_imp'] = predict_df['var_imp'].fillna(median_var_imp)

# Compute weights
predict_df['weight_export'] = predict_df.apply(
    lambda r: compute_baci_weight(r['var_exp'], r['var_imp']), axis=1
)

print(f"\nReconciliation weights (on export value):")
print(predict_df['weight_export'].describe())

# --- Step 3: Weighted reconciliation ---
def reconcile_gravity(row):
    """
    Reconcile using BACI-style weights:
    V_reconciled = w · V_export + (1-w) · V_import_fob
    """
    v_exp = row['export_value']
    v_imp = row['import_value_fob_gravity']
    w = row['weight_export']

    if pd.notna(v_exp) and pd.notna(v_imp):
        return w * v_exp + (1 - w) * v_imp
    elif pd.notna(v_exp):
        return v_exp
    elif pd.notna(v_imp):
        return v_imp
    else:
        return None

predict_df['reconciled_value_gravity'] = predict_df.apply(reconcile_gravity, axis=1)

# Data source indicator
predict_df['data_source'] = np.where(
    predict_df['export_value'].notna() & predict_df['import_value_fob_gravity'].notna(),
    'both',
    np.where(predict_df['export_value'].notna(), 'export_only', 'import_only')
)

# Summary
p2_summary = predict_df.groupby('data_source').agg(
    n=('reconciled_value_gravity', 'count'),
    total=('reconciled_value_gravity', 'sum')
).reset_index()
print(f"\n✓ Prong 2 reconciliation complete:")
print(p2_summary.to_string(index=False))

total_p2 = predict_df['reconciled_value_gravity'].sum()
print(f"\nTotal reconciled trade (Prong 2): ${total_p2/1e12:.2f} trillion")

## 8. Comparison & Output

In [ ]:
print("=" * 60)
print("COMPARISON: TIERED (PRONG 1) vs GRAVITY (PRONG 2)")
print("=" * 60)

# Bring Prong 1 results into pandas
p1_df = con.execute("""
    SELECT exporter, importer, hs4,
           export_value, import_value,
           cif_rate_tiered,
           import_value_fob_tiered,
           reconciled_value_tiered,
           data_source
    FROM prong1_reconciled
""").df()

# Merge for comparison (on flows present in both)
comparison = p1_df.merge(
    predict_df[['exporter', 'importer', 'hs4',
                'cif_rate_predicted', 'import_value_fob_gravity',
                'reconciled_value_gravity', 'weight_export']],
    on=['exporter', 'importer', 'hs4'],
    how='inner'
)

print(f"\nFlows in both prongs: {len(comparison):,}")

# --- Aggregate comparison ---
print(f"\nTotal reconciled trade:")
print(f"  Prong 1 (tiered):  ${comparison['reconciled_value_tiered'].sum()/1e12:.3f} T")
print(f"  Prong 2 (gravity): ${comparison['reconciled_value_gravity'].sum()/1e12:.3f} T")

# --- CIF rate comparison ---
print(f"\nCIF rate comparison:")
print(f"  Tiered (fixed):    mean={comparison['cif_rate_tiered'].mean():.4f}")
print(f"  Gravity (predicted): mean={comparison['cif_rate_predicted'].mean():.4f}")

# --- Flow-level differences ---
comparison['pct_diff'] = (
    (comparison['reconciled_value_gravity'] - comparison['reconciled_value_tiered'])
    / comparison['reconciled_value_tiered'] * 100
)

print(f"\nFlow-level % difference (gravity vs tiered):")
print(f"  Mean:   {comparison['pct_diff'].mean():.2f}%")
print(f"  Median: {comparison['pct_diff'].median():.2f}%")
print(f"  Std:    {comparison['pct_diff'].std():.2f}%")
print(f"  IQR:    [{comparison['pct_diff'].quantile(0.25):.2f}%, "
      f"{comparison['pct_diff'].quantile(0.75):.2f}%]")

# --- Correlation ---
corr = comparison[['reconciled_value_tiered', 'reconciled_value_gravity']].corr().iloc[0, 1]
print(f"\nCorrelation between prong 1 and prong 2 reconciled values: {corr:.6f}")

# --- By tier breakdown ---
# Tier-level comparison: avg tiered rate vs avg gravity rate per tier
tier_comparison = con.execute("""
    SELECT
        p1.geo_tier,
        COUNT(*) AS n,
        AVG(p1.cif_rate_tiered) AS avg_tiered_rate
    FROM prong1_adjusted p1
    GROUP BY p1.geo_tier
    ORDER BY n DESC
""").df()

# Merge gravity predictions with tier info
tier_gravity_df = con.execute("""
    SELECT p1.exporter, p1.importer, p1.hs4, p1.geo_tier, p1.cif_rate_tiered
    FROM prong1_adjusted p1
""").df().merge(
    predict_df[['exporter', 'importer', 'hs4', 'cif_rate_predicted']],
    on=['exporter', 'importer', 'hs4'],
    how='inner'
)

if len(tier_gravity_df) > 0:
    tier_gravity_avg = tier_gravity_df.groupby('geo_tier').agg(
        avg_gravity_rate=('cif_rate_predicted', 'mean'),
        n=('cif_rate_predicted', 'count')
    ).reset_index()

    tier_merged = tier_comparison.merge(tier_gravity_avg, on='geo_tier', suffixes=('_tiered', '_gravity'))
    print(f"\nCIF rates by geographic tier — tiered vs gravity:")
    print(tier_merged.to_string(index=False))

In [ ]:
import os

OUTPUT_DIR = os.path.join(DRIVE_ROOT, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Save Prong 1 ---
p1_output_path = os.path.join(OUTPUT_DIR, f"prong1_tiered_reconciled_{TARGET_YEAR}.csv")
p1_df.to_csv(p1_output_path, index=False)
print(f"✓ Prong 1 saved: {p1_output_path}")
print(f"  Rows: {len(p1_df):,}")

# --- Save Prong 2 ---
p2_cols = ['exporter', 'importer', 'hs4', 'export_value', 'import_value',
           'cif_rate_predicted', 'import_value_fob_gravity',
           'weight_export', 'reconciled_value_gravity', 'data_source']
p2_output = predict_df[p2_cols].copy()
p2_output_path = os.path.join(OUTPUT_DIR, f"prong2_gravity_reconciled_{TARGET_YEAR}.csv")
p2_output.to_csv(p2_output_path, index=False)
print(f"\n✓ Prong 2 saved: {p2_output_path}")
print(f"  Rows: {len(p2_output):,}")

# --- Save comparison summary ---
comp_path = os.path.join(OUTPUT_DIR, f"prong_comparison_{TARGET_YEAR}.csv")
comparison.to_csv(comp_path, index=False)
print(f"\n✓ Comparison saved: {comp_path}")

# --- Save regression coefficients ---
coeff_df = pd.DataFrame({
    'variable': coeffs.index,
    'coefficient': coeffs.values,
    'std_err': model_clean.bse.values,
    'pvalue': model_clean.pvalues.values
})
coeff_path = os.path.join(OUTPUT_DIR, f"gravity_coefficients_{TARGET_YEAR}.csv")
coeff_df.to_csv(coeff_path, index=False)
print(f"\n✓ Coefficients saved: {coeff_path}")

# --- Also save to local Colab disk (faster access) ---
p1_df.to_csv(f"/content/prong1_reconciled_{TARGET_YEAR}.csv", index=False)
p2_output.to_csv(f"/content/prong2_reconciled_{TARGET_YEAR}.csv", index=False)
print(f"\n✓ Local copies saved to /content/")

print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE — {TARGET_YEAR}")
print(f"{'='*60}")
print(f"  Prong 1 (tiered):  {len(p1_df):,} flows, ${p1_df['reconciled_value_tiered'].sum()/1e12:.2f}T")
print(f"  Prong 2 (gravity): {len(p2_output):,} flows, ${p2_output['reconciled_value_gravity'].sum()/1e12:.2f}T")
print(f"\nOutputs saved to: {OUTPUT_DIR}")
print(f"\nNext steps:")
print(f"  1. Review Prong 1 vs Prong 2 comparison")
print(f"  2. Validate gravity coefficients against BACI literature")
print(f"  3. Extend to 2024 and 2025 data")